# RAG and Agentic Search with the Claude API

**Live online course — instructor walkthrough notebook**

This notebook builds a complete Retrieval-Augmented Generation pipeline from scratch — chunking, embeddings, vector search, BM25, hybrid retrieval, reranking, and contextual retrieval — so students see the moving parts before reaching for a heavyweight library. Each section has:

- **Lecture notes** (markdown) — what to explain on the screen.
- **Demo code** — cells to run live so students see real output.
- **🏫 During class** callouts — specific instructor actions, questions to ask, and variations to try.

---

## Agenda

1. Introducing Retrieval-Augmented Generation
2. Text Chunking Strategies
3. Text Embeddings
4. The Full RAG Flow
5. Implementing the RAG Flow
6. BM25 Lexical Search
7. A Multi-Index RAG Pipeline
8. Reranking Results
9. Contextual Retrieval
10. Recap + practice exercises

## 0. Setup (do this before class starts)

1. Install dependencies:
   ```bash
   pip install anthropic voyageai python-dotenv
   ```
2. Create a file named `.env` in the same directory as this notebook containing:
   ```
   ANTHROPIC_API_KEY="sk-ant-...your-key..."
   VOYAGE_API_KEY="pa-...your-voyage-key..."
   ```
   Voyage AI is Anthropic's recommended embedding provider. Sign up at [voyageai.com](https://www.voyageai.com/) — the free tier is plenty for this notebook.
3. Add `.env` to `.gitignore` so it is never committed to version control.
4. The course corpus lives at `rag_and_agentic_search/report.md`. All later cells assume the notebook is launched from the repo root.

> **🏫 During class:** Before running the first cell, open `.env` and show students what it looks like (blur both keys). Emphasize: **never paste an API key directly into a notebook cell** — `.env` + `python-dotenv` keeps secrets out of source control. Confirm Voyage AI is approved for the engagement before running any embedding cells.

In [ ]:
# Install packages (uncomment if not already installed)
# %pip install anthropic voyageai python-dotenv

1. `load_dotenv()` — reads the `.env` file sitting next to the notebook and copies every `KEY="value"` line into the process's environment variables. After this runs, both `ANTHROPIC_API_KEY` and `VOYAGE_API_KEY` are available to the SDKs.
2. `client = anthropic.Anthropic()` — Anthropic SDK client. With no arguments it picks up `ANTHROPIC_API_KEY` from the environment.
3. `voyage = voyageai.Client()` — Voyage AI client for embedding generation. It picks up `VOYAGE_API_KEY` the same way.
4. `model = "claude-sonnet-4-6"` — the default workhorse for the course. Swap this one line and every Claude demo switches model.
5. The four `print(...)` lines act as a pre-class sanity check. If anything is wrong, these tell you immediately — most commonly `Voyage key loaded: False`, which means a missing or misnamed `.env`.

In [ ]:
from dotenv import load_dotenv
import anthropic
import voyageai
import os

load_dotenv()  # loads ANTHROPIC_API_KEY and VOYAGE_API_KEY from .env

client = anthropic.Anthropic()      # picks up ANTHROPIC_API_KEY
voyage = voyageai.Client()          # picks up VOYAGE_API_KEY

# We'll use Sonnet 4.6 as the default workhorse model for the course.
model = "claude-sonnet-4-6"

print("SDK version:", anthropic.__version__)
print("Model:", model)
print("Anthropic key loaded:", bool(os.getenv("ANTHROPIC_API_KEY")))
print("Voyage key loaded:", bool(os.getenv("VOYAGE_API_KEY")))

### Shared helpers (used by every later section that calls Claude)

We define one `chat()` helper here and reuse it for the rest of the notebook — reranking and contextual retrieval both call it. Keeping a single helper makes it visually obvious in later cells that *we are only varying one input*: the prompt, the temperature, or the stop sequence.

In [ ]:
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})
    return messages

def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})
    return messages

def chat(messages, system=None, temperature=1.0, stop_sequences=None):
    """Send messages to Claude and return the assistant text."""
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature,
    }
    if system is not None:
        params["system"] = system
    if stop_sequences is not None:
        params["stop_sequences"] = stop_sequences
    response = client.messages.create(**params)
    return response.content[0].text

---
# 1. Introducing Retrieval-Augmented Generation

**RAG** is a pattern for letting a language model answer questions about documents that are too large — or too numerous — to fit in the prompt.

### The two approaches

| Approach | Idea | Reality |
|---|---|---|
| **Direct (stuff-the-prompt)** | Paste the entire document into the request. | Fast to build. Hits hard token limits, costs scale linearly with size, and answer quality drops as prompts get longer. |
| **RAG** | (1) split the document into chunks; (2) at query time, pull only the few chunks that look relevant. | More plumbing up front. Smaller prompts, lower cost, faster responses, scales to many documents. |

### Why care

Most real corpora — internal wikis, support tickets, regulatory filings, code — are far past any model's effective context. RAG turns *"will this fit?"* into *"can we find the right ~5 KB?"*

### Trade-offs to flag up front

- More moving parts: chunker, embedder, vector store, retriever, plus the LLM call.
- Retrieval quality is now its own quality metric — bad chunks → bad answers.
- "Relevance" is fuzzy and use-case dependent, which is why this whole notebook exists.

### Demo: how big is the source corpus?

Before we touch RAG, we want students to feel why we cannot just paste the document into a prompt. This cell loads `rag_and_agentic_search/report.md`, prints its character count, the number of `##` sections, and a tiny preview. The instructor should point out that even a friendly internal report is already non-trivial — a real corpus is 100× larger.

In [ ]:
from pathlib import Path

REPORT_PATH = Path("rag_and_agentic_search/report.md")
text = REPORT_PATH.read_text()

print("Source path :", REPORT_PATH)
print("Characters  :", len(text))
print("Words (~)   :", len(text.split()))
print("## sections :", text.count("\n## "))
print("\n--- First 400 chars ---")
print(text[:400])

> **🏫 During class:**
> 1. Run the cell and read out the character count.
> 2. Talking point: *"This single report is already meaningful. Now imagine 5,000 of these. Stuffing the prompt is not a strategy — it's a deferred bug."*
> 3. Ask the room: *"What goes wrong first when prompts get huge — cost, latency, or accuracy?"* (All three, but accuracy is the surprising answer.)

---
# 2. Text Chunking Strategies

**Chunking** is how we cut a document into the pieces we will later search over. It is the single biggest lever on RAG quality — bad chunks return irrelevant context no matter how good the rest of the pipeline is.

### Three strategies

| Strategy | How it cuts | Best when | Watch out for |
|---|---|---|---|
| **Size-based** | Fixed-length character windows, often with overlap. | You have no formatting guarantees. Most common in production. | Cuts mid-word, mid-sentence; chunks read out of context. **Overlap** softens this at the cost of duplication. |
| **Structure-based** | Split on headers, paragraphs, or sections. | Documents are reliably formatted (Markdown, HTML). | Falls apart on inconsistent or partial formatting. |
| **Semantic** | Group consecutive sentences by similarity. | High-stakes, topic-heterogeneous text. | Most complex; slowest; depends on a sentence-level embedding model. |

### Picking one

- **Chunk by character** is the safe fallback — works on anything.
- **Chunk by sentence** is a decent middle ground if your sentence detection is reliable.
- **Chunk by section** wins on quality when you can guarantee structure.

There is no universal best. Match the chunker to what your corpus actually looks like.

### Demo: same document, three chunkers

We define three chunkers (character / sentence / section) and run them on the same `report.md`. Watch the **count** and the **first chunk preview** for each — character chunks are mid-sentence, sentence chunks are clean but lose section context, and section chunks line up with the document's `##` headers. Tweak `chunk_size` or the regex to see how brittle each strategy is.

In [ ]:
import re

def chunk_by_char(text, chunk_size=500, chunk_overlap=50):
    chunks = []
    start_idx = 0
    while start_idx < len(text):
        end_idx = min(start_idx + chunk_size, len(text))
        chunks.append(text[start_idx:end_idx])
        start_idx = end_idx - chunk_overlap if end_idx < len(text) else len(text)
    return chunks

def chunk_by_sentence(text, max_sentences=4, overlap=1):
    sentences = re.split(r"(?<=[.!?])\s+", text)
    chunks, start = [], 0
    while start < len(sentences):
        end = min(start + max_sentences, len(sentences))
        chunks.append(" ".join(sentences[start:end]))
        start += max_sentences - overlap
    return chunks

def chunk_by_section(document_text):
    return re.split(r"\n## ", document_text)

char_chunks    = chunk_by_char(text)
sent_chunks    = chunk_by_sentence(text)
section_chunks = chunk_by_section(text)

for name, chunks in [("char", char_chunks), ("sentence", sent_chunks), ("section", section_chunks)]:
    print(f"--- {name:>8}: {len(chunks):>3} chunks ---")
    print(repr(chunks[0][:200]))
    print()

> **🏫 During class:**
> 1. Run the cell. Read the `repr()` for each strategy out loud — the quotes make the cut points visible.
> 2. Talking point: *"The character chunker can split mid-word. The sentence chunker can split mid-section. The section chunker is clean — but only because this report has well-formed `##` headers. Change one heading style and section chunking breaks."*
> 3. Variation to try live: drop `chunk_overlap` to `0` and re-run — the boundary now becomes a hard cut. Or change the section regex to `\n# ` and watch the count drop.

---
# 3. Text Embeddings

An **embedding** is a numerical fingerprint of meaning. An embedding model takes a string of text and returns a long list of numbers in roughly `[-1, 1]`. Each number scores some unknown latent feature of the text. We do not interpret individual numbers — we compare *whole vectors* to see how similar two pieces of text are.

### Why this matters for RAG

Once every chunk has an embedding, finding *"chunks related to a user's question"* becomes *"chunks whose embedding is close to the question's embedding"* — a math operation, not a keyword match. That is **semantic search**, and it is the search mechanism RAG needs.

### Provider

Anthropic recommends [Voyage AI](https://www.voyageai.com/) for production embeddings. Free tier, separate account, drop-in SDK. We use `voyage-3-large` throughout this notebook — it's the current best-quality general-purpose Voyage model.

### Demo: distance is meaning

We embed three short strings — two about the same topic, one about something different — and print the pairwise cosine distances. The two related strings should be visibly closer than either is to the unrelated one. Have the class predict the ordering before the cell runs.

In [ ]:
import math

def generate_embedding(chunks, embed_model="voyage-3-large", input_type="query"):
    is_list = isinstance(chunks, list)
    inputs = chunks if is_list else [chunks]
    result = voyage.embed(inputs, model=embed_model, input_type=input_type)
    return result.embeddings if is_list else result.embeddings[0]

def cosine_distance(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    mag = math.sqrt(sum(x * x for x in a)) * math.sqrt(sum(y * y for y in b))
    return 1.0 - (dot / mag if mag else 0.0)

a = "The engineering team shipped a new logging system last quarter."
b = "Our backend group rolled out improved observability tooling."
c = "A medieval recipe for fermented honey wine."

va, vb, vc = generate_embedding([a, b, c])
print(f"len(embedding)        = {len(va)}")
print(f"distance(a, b) related = {cosine_distance(va, vb):.4f}")
print(f"distance(a, c) unrelated = {cosine_distance(va, vc):.4f}")
print(f"distance(b, c) unrelated = {cosine_distance(vb, vc):.4f}")

> **🏫 During class:**
> 1. Before running, ask: *"Which two strings will be closest? Which two will be furthest?"*
> 2. Run the cell. Talking point: *"Notice the embedding length — over 1,000 dimensions. We never read those numbers individually. We only compare full vectors. Distance is a stand-in for 'aboutness'."*
> 3. Variation: swap string `c` for *"The platform team improved its incident response playbook."* — distances should compress dramatically. That collapse is exactly what makes semantic search work for paraphrases.

---
# 4. The Full RAG Flow

Putting chunking and embeddings together gives us a 7-step pipeline:

**Pre-processing (offline, once per document):**
1. **Chunk** the document.
2. **Embed** every chunk.
3. **Normalize** vector magnitudes to 1.0 (Voyage does this automatically).
4. **Store** the embeddings in a vector database, with the original text kept as metadata.

**Retrieval (online, per query):**
5. **Embed** the user's question with the same model.
6. **Search** the store for the closest stored embeddings.
7. **Assemble** a prompt: original question + retrieved chunks → send to Claude.

### Cosine similarity vs cosine distance

| Metric | Range | Closer match means |
|---|---|---|
| Cosine **similarity** | -1 to 1 | Closer to **1** |
| Cosine **distance** = 1 − similarity | 0 to 2 | Closer to **0** |

We use **distance** in the demo class below because it sorts naturally (smallest first = best match).

### Demo: a tiny VectorIndex from scratch

This cell defines a hand-rolled `VectorIndex` so students see exactly what a vector database does — `.add_vector()` stores a vector with metadata, `.search()` ranks every stored vector by cosine distance to the query and returns the top `k`. In production you would reach for FAISS, pgvector, or a managed service. The data structure is identical; this is just the readable version. We also exercise it on three toy strings before pointing it at `report.md` in the next section.

In [ ]:
from typing import Optional, Any, List, Dict, Tuple

class VectorIndex:
    def __init__(self, embedding_fn=None):
        self.vectors: List[List[float]] = []
        self.documents: List[Dict[str, Any]] = []
        self._dim: Optional[int] = None
        self._embedding_fn = embedding_fn

    def add_vector(self, vector, document):
        if not self.vectors:
            self._dim = len(vector)
        elif len(vector) != self._dim:
            raise ValueError(f"Expected dim {self._dim}, got {len(vector)}")
        self.vectors.append(list(vector))
        self.documents.append(document)

    def add_document(self, document):
        if not self._embedding_fn:
            raise ValueError("No embedding_fn provided.")
        self.add_vector(self._embedding_fn(document["content"]), document)

    def add_documents(self, documents):
        # Bulk-embed in one Voyage call to avoid rate limits.
        if not self._embedding_fn:
            raise ValueError("No embedding_fn provided.")
        contents = [d["content"] for d in documents]
        vectors = self._embedding_fn(contents)
        for v, d in zip(vectors, documents):
            self.add_vector(v, d)

    def search(self, query, k=2):
        q_vec = self._embedding_fn(query) if isinstance(query, str) else query
        scored = []
        for v, d in zip(self.vectors, self.documents):
            scored.append((cosine_distance(q_vec, v), d))
        scored.sort(key=lambda x: x[0])
        return [(d, dist) for dist, d in scored[:k]]

    def __len__(self):
        return len(self.vectors)

# Quick smoke test on the three strings from §3
demo_index = VectorIndex(embedding_fn=generate_embedding)
demo_index.add_documents([
    {"content": a, "label": "engineering"},
    {"content": b, "label": "engineering"},
    {"content": c, "label": "unrelated"},
])
for doc, dist in demo_index.search("who improved logging?", k=3):
    print(f"{dist:.4f}  {doc['label']:>12}  {doc['content'][:60]}")

> **🏫 During class:**
> 1. Run the cell. Confirm the two `engineering`-labelled docs come back ahead of the unrelated one.
> 2. Talking point: *"There is nothing magical here. A vector store is a list of vectors plus a sort. The 'database' is the index, not the algorithm."*
> 3. Variation to try: query for *"medieval beverage"* — the unrelated string should now win. Same store, different question, different ranking.

---
# 5. Implementing the Full RAG Flow

Now we glue everything together against the real corpus. Five concrete steps:

1. **Chunk** `report.md` by section.
2. **Embed** every chunk (single bulk call to Voyage).
3. **Populate** a `VectorIndex` with `(embedding, {content: chunk})` pairs.
4. **Embed the user query**.
5. **Search** for the top-k closest chunks and inspect them.

Storing the original text as metadata is what makes step 5 useful — without it, all you get back is a vector.

### Demo: end-to-end retrieval against `report.md`

We chunk by section, build the index, then run a real query: *"what did the software engineering department do last year?"* The cell prints the top-2 matches with their cosine distances. You should see two related sections of the report come back (low distance ≈ good match) and unrelated sections sit further away. Swap the question to see the ranking shift.

In [ ]:
# 1. Chunk
chunks = chunk_by_section(text)
print(f"Chunked into {len(chunks)} sections.")

# 2 + 3. Embed every chunk and populate the index in one bulk call.
store = VectorIndex(embedding_fn=generate_embedding)
store.add_documents([{"content": c} for c in chunks])
print(f"Index size: {len(store)} vectors.")

# 4 + 5. Embed query and search.
user_query = "what did the software engineering department do last year?"
results = store.search(user_query, k=2)

for i, (doc, dist) in enumerate(results, start=1):
    preview = doc["content"][:160].replace("\n", " ")
    print(f"\n[{i}] distance={dist:.4f}")
    print(f"    {preview}...")

> **🏫 During class:**
> 1. Run the cell. Read the section header from each match — the engineering-related sections should win.
> 2. Talking point: *"This is the entire RAG pipeline in 15 lines. Everything else in the rest of the notebook is making this more accurate."*
> 3. Variation: change `user_query` to *"what was the marketing budget?"* and re-run. Different sections rise. Then ask: *"What if the answer was paraphrased without the word 'engineering'? Would semantic search still find it?"* (Yes — and the next section shows where it can fail.)

---
# 6. BM25 Lexical Search

**BM25** (Best Match 25) is a classical lexical search algorithm. Where embeddings score *meaning*, BM25 scores *exact term overlap* — weighted so that rare terms count more than common ones.

### Why we need it alongside embeddings

Semantic search can miss specific identifiers — a part number, a person's name, an error code — because the embedding model has no signal that the exact token matters. BM25 catches these. The standard production pattern is a **hybrid retriever**: run both, merge the rankings.

### What BM25 actually does

1. **Tokenize** the query (lowercase, strip punctuation, split on non-word characters).
2. **Document frequency** — for each term, count how many chunks contain it.
3. **IDF weight** — rare terms get higher weight; common terms (`the`, `a`) get near-zero weight.
4. **Score** each chunk by how often it contains the higher-weighted query terms (with a saturation curve so a 100× match isn't 100× the score).
5. Return the top-k chunks by score.

### Demo: a hand-rolled BM25Index

Same `add_document` / `search` shape as `VectorIndex`, so a retriever can wrap both. We score the same query against the same chunks and print the top-2. Compare the matches to what the vector search returned — sometimes they agree, sometimes they don't, and where they disagree is exactly the value of running both.

In [ ]:
from collections import Counter
from typing import Callable

class BM25Index:
    def __init__(self, k1=1.5, b=0.75, tokenizer: Optional[Callable[[str], List[str]]] = None):
        self.documents: List[Dict[str, Any]] = []
        self._tokens: List[List[str]] = []
        self._lens: List[int] = []
        self._df: Dict[str, int] = {}
        self._idf: Dict[str, float] = {}
        self._avg_len = 0.0
        self._built = False
        self.k1, self.b = k1, b
        self._tokenizer = tokenizer or self._default_tokenizer

    @staticmethod
    def _default_tokenizer(text):
        return [t for t in re.split(r"\W+", text.lower()) if t]

    def add_document(self, document):
        toks = self._tokenizer(document["content"])
        self.documents.append(document)
        self._tokens.append(toks)
        self._lens.append(len(toks))
        for t in set(toks):
            self._df[t] = self._df.get(t, 0) + 1
        self._built = False

    def add_documents(self, documents):
        for d in documents:
            self.add_document(d)

    def _build(self):
        N = len(self.documents)
        if N == 0:
            self._avg_len, self._idf, self._built = 0.0, {}, True
            return
        self._avg_len = sum(self._lens) / N
        self._idf = {t: math.log(((N - f + 0.5) / (f + 0.5)) + 1) for t, f in self._df.items()}
        self._built = True

    def search(self, query, k=2):
        if not self._built:
            self._build()
        q_toks = self._tokenizer(query)
        scored = []
        for i, toks in enumerate(self._tokens):
            counts = Counter(toks)
            score = 0.0
            for t in q_toks:
                if t not in self._idf:
                    continue
                tf = counts.get(t, 0)
                num = self._idf[t] * tf * (self.k1 + 1)
                denom = tf + self.k1 * (1 - self.b + self.b * (self._lens[i] / self._avg_len))
                score += num / (denom + 1e-9)
            if score > 0:
                scored.append((score, self.documents[i]))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [(d, s) for s, d in scored[:k]]

    def __len__(self):
        return len(self.documents)

bm25 = BM25Index()
bm25.add_documents([{"content": c} for c in chunks])
print(f"BM25 index size: {len(bm25)}")

for i, (doc, score) in enumerate(bm25.search(user_query, k=2), start=1):
    preview = doc["content"][:160].replace("\n", " ")
    print(f"\n[{i}] bm25_score={score:.4f}")
    print(f"    {preview}...")

> **🏫 During class:**
> 1. Run the cell. Compare the BM25 top-2 to the vector search top-2 from §5.
> 2. Talking point: *"Notice that BM25 has no concept of 'meaning'. It only knows that the rare query terms appear here more often than elsewhere. That is its weakness — and its strength."*
> 3. Variation to try: query *"INC-2023-04"* (a fake incident ID) — BM25 will pin it instantly, the vector index will struggle. Then query *"backend reliability work"* — the vector index wins because there is no exact term overlap.

---
# 7. A Multi-Index RAG Pipeline

We have two retrievers with the same shape — `add_document(s)` and `search(query, k)`. A `Retriever` wrapper forwards a query to every index it owns, then merges their rankings into a single ordered list.

### Reciprocal Rank Fusion (RRF)

Each index votes by the *rank* it gave a document, not its raw score (vector distances and BM25 scores are not directly comparable). The fused score for a document is:

```
rrf_score(doc) = Σ over indexes  1 / (k_rrf + rank_in_index)
```

Documents that placed near the top in *any* index get a meaningful contribution; documents that placed top in *both* indexes win.

### Why this is a clean abstraction

- **Modular** — each index is independent; you can swap implementations without touching the others.
- **Standardized API** — `Retriever` doesn't care whether an index is BM25, dense vectors, or something exotic.
- **Extensible** — add a third index (say, a knowledge graph lookup) and `Retriever` just forwards to it.

### Demo: hybrid retrieval beats either index alone

We build a `Retriever(bm25, vector_index)` over the same chunks, then run the engineering query and compare its top-2 to what each index produced individually. The combined ranking should pick up matches both indexes agreed on. We also run a query designed to expose where pure vector search loses (an exact term that doesn't paraphrase well) so students see the hybrid pulling in BM25's strength.

In [ ]:
from typing import Protocol

class SearchIndex(Protocol):
    def add_documents(self, documents: List[Dict[str, Any]]) -> None: ...
    def search(self, query: Any, k: int = 1) -> List[Tuple[Dict[str, Any], float]]: ...

class Retriever:
    def __init__(self, *indexes: SearchIndex):
        if not indexes:
            raise ValueError("At least one index required.")
        self._indexes = list(indexes)

    def add_documents(self, documents):
        for idx in self._indexes:
            idx.add_documents(documents)

    def search(self, query, k=2, k_rrf=60):
        all_results = [idx.search(query, k=k * 5) for idx in self._indexes]
        ranks: Dict[int, Dict[str, Any]] = {}
        for idx_i, results in enumerate(all_results):
            for rank, (doc, _) in enumerate(results):
                key = id(doc)
                slot = ranks.setdefault(key, {"doc": doc, "r": [float("inf")] * len(self._indexes)})
                slot["r"][idx_i] = rank + 1
        scored = [
            (slot["doc"], sum(1.0 / (k_rrf + r) for r in slot["r"] if r != float("inf")))
            for slot in ranks.values()
        ]
        scored.sort(key=lambda x: x[1], reverse=True)
        return scored[:k]

# Build a fresh hybrid retriever over the same chunks.
vector_index = VectorIndex(embedding_fn=generate_embedding)
bm25_index   = BM25Index()
retriever    = Retriever(bm25_index, vector_index)
retriever.add_documents([{"content": c} for c in chunks])

for query in [user_query, "backend reliability work"]:
    print(f"\n=== HYBRID for: {query!r} ===")
    for i, (doc, rrf) in enumerate(retriever.search(query, k=2), start=1):
        preview = doc["content"][:140].replace("\n", " ")
        print(f"[{i}] rrf={rrf:.4f}  {preview}...")

> **🏫 During class:**
> 1. Run the cell. For each query, read the top-2 RRF result and recall how the single-index runs ranked them in §5 and §6.
> 2. Talking point: *"RRF is doing voting, not averaging. A document only needs to land high in one index to get a meaningful score — but documents that land high in both win."*
> 3. Variation: instantiate a `Retriever(vector_index)` only (one index). Same RRF code, but now you've reduced it to a single-source ranker. The abstraction holds.

---
# 8. Reranking Results

**Reranking** is a post-processing step: take the top-N candidates from the retriever, hand them to an LLM, and ask the LLM to reorder them by true relevance to the query.

### Why reranking helps

Both BM25 and embeddings are *local* signals — exact-term overlap and approximate semantic similarity. They miss intent. *"What did the ENG team do with incident 2023?"* should clearly retrieve the software engineering section over the cybersecurity section, but a hybrid retriever might still split the call. An LLM reads the candidates and the question together and judges directly.

### The trade-off

- **More accurate** — the model knows what "engineering" usually means, what "incident" implies, who "the team" probably refers to.
- **Slower and pricier** — every query now triggers an extra LLM call. Run reranking only on the top-N (e.g. 10) candidates, not on the whole corpus.

### Implementation tricks

- Reference candidates by **ID**, not by full text — the model returns IDs, you map back to documents. Saves tokens.
- **Pre-fill** the assistant turn with the start of a JSON array and use a **stop sequence** to close it. That forces the model to emit a parseable list and nothing else.

### Demo: hybrid → rerank

We take the top-5 hybrid hits for an intent-heavy query, hand them to Claude with explicit instructions, and let it produce a JSON array of IDs in order of true relevance. The cell prints the hybrid order and the reranked order side by side. The reranked top should match human judgment of which section actually answers the question.

In [ ]:
import json

def rerank(query, candidates, top_k=3):
    """Use Claude to reorder retriever candidates by relevance to the query."""
    # Build a compact id-keyed list so we don't pay for repeated text.
    items = [{"id": i, "content": c["content"][:600]} for i, c in enumerate(candidates)]
    prompt = (
        "You are a relevance reranker. Given a user query and a list of candidate "
        "documents (each with an id), return ONLY the ids of the most relevant "
        f"documents, ordered most relevant first, as a JSON array of at most {top_k} ids.\n\n"
        f"Query: {query}\n\n"
        f"Candidates: {json.dumps(items)}"
    )
    messages = [
        {"role": "user", "content": prompt},
        # Pre-fill: force the model to emit a JSON array and stop at the closing bracket.
        {"role": "assistant", "content": "["},
    ]
    raw = chat(messages, temperature=0.0, stop_sequences=["]"])
    ids = json.loads("[" + raw + "]")
    return [candidates[i] for i in ids if 0 <= i < len(candidates)]

intent_query = "What did the engineering team do with incident 2023?"
hybrid_top   = [doc for doc, _ in retriever.search(intent_query, k=5)]
reranked     = rerank(intent_query, hybrid_top, top_k=3)

print("--- HYBRID order (top 5) ---")
for i, doc in enumerate(hybrid_top, start=1):
    print(f"[{i}] {doc['content'][:100]!r}...")

print("\n--- RERANKED order (top 3) ---")
for i, doc in enumerate(reranked, start=1):
    print(f"[{i}] {doc['content'][:100]!r}...")

> **🏫 During class:**
> 1. Run the cell and read the two orderings out loud.
> 2. Talking point: *"The pre-fill `"["` plus stop sequence `"]"` is a common trick — we are constraining the model to emit a list, period. No prose, no explanation, no fenced block. The output goes straight into `json.loads`."*
> 3. Variation: bump `temperature` to `1.0` and re-run a few times. Show that with `temperature=0.0` the rerank is deterministic across runs — exactly what you want for a deterministic retrieval pipeline.

---
# 9. Contextual Retrieval

**Contextual Retrieval** is a pre-processing trick: before embedding (and indexing in BM25), we *prepend* a short, model-generated context blurb to each chunk. The blurb explains how this chunk relates to the surrounding document.

### Why this matters

When you split a document into chunks, a single chunk may say *"the team prioritized this for Q3"* — but who is *the team*? Which Q3? The original document had that context; the chunk does not. Without contextual retrieval, the embedding for this chunk is ambiguous, and BM25 can't match cross-references either.

### Process

1. For each chunk, send `(chunk, surrounding document)` to Claude with a prompt asking for a 1–2 sentence situating context.
2. Receive the contextual blurb.
3. Build the **contextualized chunk** = `context_blurb + "\n\n" + original_chunk`.
4. Feed the contextualized chunk into the vector and BM25 indexes.
5. At query time, search as usual. The original chunk text is what you ultimately surface to Claude — the contextual blurb is only there to help retrieval find the right chunk.

### When the source document is too big

If the full document blows past your prompt budget, build a *selective* surrounding instead:

- the first 1–3 chunks (gives the abstract / framing),
- the chunks immediately before the target chunk (gives local context),
- skip everything else.

This trades completeness for usable prompt size and keeps the latency cost bounded.

### Demo: a contextualized chunk

We take a single chunk from the middle of `report.md`, ask Claude to write a one-sentence situating context, and print **(a)** the raw chunk, **(b)** the generated context, **(c)** the joined contextualized chunk that would actually go into the index. In production you'd loop this over every chunk before indexing — we do just one here so the contrast is readable on screen.

In [ ]:
def add_context(chunk, source_text):
    """Generate a short situating context for a chunk and prepend it."""
    prompt = (
        "You will be given a full source document and one chunk extracted from it. "
        "Write a SHORT 1-2 sentence context that situates the chunk inside the larger "
        "document. Do not summarize the chunk itself; only provide context that the "
        "chunk is missing on its own.\n\n"
        f"<source>\n{source_text}\n</source>\n\n"
        f"<chunk>\n{chunk}\n</chunk>"
    )
    context = chat([{"role": "user", "content": prompt}], temperature=0.0).strip()
    return f"{context}\n\n{chunk}"

# Pick a middle chunk so the situating context has something to anchor on.
target_chunk = chunks[len(chunks) // 2]
contextualized = add_context(target_chunk, text)

print("--- ORIGINAL CHUNK (first 300 chars) ---")
print(target_chunk[:300])
print("\n--- CONTEXTUALIZED CHUNK (first 500 chars) ---")
print(contextualized[:500])

> **🏫 During class:**
> 1. Run the cell. Read the original chunk first, then the generated context line.
> 2. Talking point: *"That extra sentence is what gives the chunk its lineage. Without it, the embedding has no idea this paragraph belongs to the engineering section, or that 'the team' refers to the platform group introduced two pages earlier."*
> 3. Question for the room: *"What's the cost trade-off here?"* (Embeddings are computed once per chunk, but contextualization adds an LLM call per chunk. Cheap at indexing time, free at query time.) Variation: re-rank §8's hybrid output after rebuilding the index with contextualized chunks — accuracy on intent-heavy queries should improve.

---
# 10. Recap + practice exercises

### Recap (run through these out loud)

- **RAG = chunk → embed → store → search → assemble.** Smaller prompts, lower cost, scales to many documents.
- **Chunking is the biggest quality lever.** Pick a strategy that matches what your corpus actually looks like.
- **Embeddings are about distance, not numbers.** A vector store is a list of vectors plus a sort.
- **BM25 catches what embeddings miss.** Specific identifiers, rare terms, exact matches.
- **Hybrid retrieval beats either index alone.** Reciprocal Rank Fusion is a clean way to merge.
- **Reranking with an LLM** improves intent-heavy queries at the cost of one extra API call.
- **Contextual retrieval** rescues chunks that lost their lineage during chunking.

### Exercises (progressively harder)

1. **Switch chunkers.** Replace `chunk_by_section` with `chunk_by_char` (or `chunk_by_sentence`) when populating the retriever. Re-run the engineering query. Did the top result change? Why?
2. **Make the retriever 3-way.** Add a second `VectorIndex` configured with a different embedding model (e.g. `voyage-3` instead of `voyage-3-large`) and pass all three indexes into `Retriever`. Same query, different fusion — does it stabilize the ranking?
3. **Wire reranking into the retriever.** Write `retriever.search_and_rerank(query, k=3)` that calls `retriever.search(query, k=10)` then `rerank()` on the result. Compare against a non-reranked run.
4. **Index every chunk contextualized.** Loop `add_context` over every chunk before indexing. Re-run the §5 and §7 queries; expect bigger gains on cross-reference questions than on direct questions.
5. **Final answer.** Wrap the whole pipeline behind a single `answer(question)` function: hybrid search → rerank → assemble a prompt with the top-3 contextualized chunks → call `chat()` and return Claude's response. That's a working RAG agent in ~30 lines on top of what you've built.

In [ ]:
# Exercise 1 scaffold — finish this live in class together.
# Build a fresh hybrid retriever using char-based chunks instead of sections,
# then compare its top hit for `user_query` to what we got in §7.
#
# new_chunks = chunk_by_char(text, chunk_size=600, chunk_overlap=80)
# vi = VectorIndex(embedding_fn=generate_embedding)
# bi = BM25Index()
# r  = Retriever(bi, vi)
# r.add_documents([{"content": c} for c in new_chunks])
# for doc, rrf in r.search(user_query, k=2):
#     print(rrf, doc["content"][:120])